# draw market - mrkt_favourite segments -> model on the most profitable segment

goals_foragst space, DRAW market only. Both teams `game_number > 8`. Two parts:

1. **Raw segment profitability** - bet the draw on *every* match in each `mrkt_favourite` segment
   (home / away / balanced), no model. Descriptive, in-sample: shows where draws are underpriced.
2. **Model on the most profitable segment** - take the segment with the best raw draw ROI, fit
   LR (PolynomialFeatures deg2 + StandardScaler) on the four running goals for/against features
   (train fold only), bet the draw where the model beats implied by `buffer`, and run the v2 gates.

Result is logged to `model_checks/results.md` via `evaluation.results_registry.upsert`.

In [1]:
GROUP = "major"
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

CONFIG = {"buffer": 0.02, "min_game_number": 8, "min_segment_train": 100}

FEATURES = [
    "homet_season_goals_for_avg", "homet_season_goals_agst_avg",
    "awayt_season_goals_for_avg", "awayt_season_goals_agst_avg",
]
SEGMENT = "mrkt_favourite"        # home / away / balanced
DRAW_OUTCOME = "t_draw_flg"
DRAW_IMPL = "b365_draw_impl"      # implied draw prob = 1 / draw odds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

_needed = FEATURES + [SEGMENT, "league", "season", DRAW_OUTCOME, DRAW_IMPL, "mrkt_draw_impl"]
matches = abt[
    (abt["homet_game_number"] > CONFIG["min_game_number"])
    & (abt["awayt_game_number"] > CONFIG["min_game_number"])
].dropna(subset=_needed).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}")
print(f"segments (mrkt_favourite): {matches[SEGMENT].value_counts().to_dict()}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']
segments (mrkt_favourite): {'balanced': 2767, 'home': 1974, 'away': 792}


In [2]:
def pooled_roi(implied, won):
    staked = implied.sum()
    return (won.sum() - staked) / staked if staked > 0 else np.nan

def roi_grouped(bets, keys):
    totals = bets.groupby(keys).agg(won=("won", "sum"), staked=("implied", "sum"))
    return (totals["won"] - totals["staked"]) / totals["staked"]

def season_folds(data):
    for season in sorted(data["season"].unique()):
        yield ~(data["season"] == season), data["season"] == season, season

def expanding_folds(data, min_train_seasons=2):
    seasons = sorted(data["season"].unique())
    for split in range(min_train_seasons, len(seasons)):
        yield data["season"].isin(seasons[:split]), data["season"] == seasons[split], seasons[split]

def build_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler(),
                         LogisticRegression(max_iter=3000))

## Part 1 - raw draw profitability per segment
Bet the draw on every match in the segment (no model, in-sample). ROI = (wins - staked) / staked under
implied-probability staking. b365 odds. Picks the most profitable segment to model in Part 2.

In [3]:
print("PART 1 - raw draw ROI per mrkt_favourite segment (bet draw on all matches, b365):")
seg_roi = {}
for seg in sorted(matches[SEGMENT].unique()):
    sub = matches[matches[SEGMENT] == seg]
    roi = pooled_roi(sub[DRAW_IMPL], sub[DRAW_OUTCOME].astype(int))
    seg_roi[seg] = roi
    frame = pd.DataFrame({"implied": sub[DRAW_IMPL], "won": sub[DRAW_OUTCOME].astype(int), "fold": sub["season"]})
    per_season = roi_grouped(frame, "fold")
    print(f"  {seg:9s} ROI {roi:+.4f}  ({len(sub)} matches)  "
          + "  ".join(f"{s}:{v:+.3f}" for s, v in per_season.items()))

BEST_SEGMENT = max(seg_roi, key=seg_roi.get)
print(f"\nmost profitable segment (raw, in-sample): {BEST_SEGMENT}  (ROI {seg_roi[BEST_SEGMENT]:+.4f})")

PART 1 - raw draw ROI per mrkt_favourite segment (bet draw on all matches, b365):
  away      ROI -0.0691  (792 matches)  2223:-0.063  2324:-0.112  2425:-0.032  2526:-0.072
  balanced  ROI -0.0433  (2767 matches)  2223:-0.097  2324:+0.050  2425:-0.075  2526:-0.047
  home      ROI -0.0366  (1974 matches)  2223:-0.101  2324:+0.068  2425:-0.094  2526:-0.011

most profitable segment (raw, in-sample): home  (ROI -0.0366)


## Part 2 - LR model on the most profitable segment
Restrict to `BEST_SEGMENT`; per fold fit LR on the training rows of that segment only, bet the draw where
`model_prob > implied + buffer`. Gates: G1 LOSO (>=3/4 seasons), G2 drop-best-league, G3 walk-forward.

In [4]:
# BEST_SEGMENT = 'balanced'

In [5]:
def collect_draw_bets(segment, folds, odds_source="b365"):
    implied_column = {"mrkt": "mrkt_draw_impl", "b365": "b365_draw_impl"}[odds_source]
    placed = []
    for train_mask, test_mask, fold_name in folds:
        train = matches[train_mask & (matches[SEGMENT] == segment)]
        test = matches[test_mask & (matches[SEGMENT] == segment)]
        if len(train) < CONFIG["min_segment_train"] or test.empty:
            continue
        model = build_model()
        model.fit(train[FEATURES], train[DRAW_OUTCOME].astype(int))
        region = test.copy()
        region["model_prob"] = model.predict_proba(region[FEATURES])[:, list(model.classes_).index(1)]
        region["implied"] = region[implied_column]
        region["won"] = region[DRAW_OUTCOME].astype(int)
        region["fold"] = fold_name
        is_bet = region["model_prob"] > region["implied"] + CONFIG["buffer"]
        placed.append(region.loc[is_bet, ["implied", "won", "league", "season", "fold"]])
    if not placed:
        return pd.DataFrame(columns=["implied", "won", "league", "season", "fold"])
    return pd.concat(placed, ignore_index=True)


_G, _R, _X = chr(27)+"[92m", chr(27)+"[91m", chr(27)+"[0m"
_paint = lambda ok: (_G+"PASS"+_X) if ok else (_R+"FAIL"+_X)
_verdict = lambda ok: "PASS" if ok else "FAIL"

bets = collect_draw_bets(BEST_SEGMENT, season_folds(matches))
walk = collect_draw_bets(BEST_SEGMENT, expanding_folds(matches))

print(f"PART 2 - LR on segment '{BEST_SEGMENT}', draw market (b365):")
if bets.empty:
    print("  no bets placed")
else:
    per_season = roi_grouped(bets, "fold")
    g1 = int((per_season > 0).sum()) >= 3
    without = {lg: pooled_roi(bets[bets["league"] != lg]["implied"], bets[bets["league"] != lg]["won"])
               for lg in bets["league"].unique()}
    g2 = min(without.values()) > 0
    wf = roi_grouped(walk, "fold")
    g3 = (not walk.empty) and bool((wf >= -0.05).all()) and not bool((wf < 0).all())
    pooled = pooled_roi(bets["implied"], bets["won"])
    spread = roi_grouped(bets, ["league", "season"]).std()

    print(f"  pooled ROI                     {pooled:+.4f}  ({len(bets)} bets)")
    print(f"  GATE 1  LOSO (>=3/4)            {_paint(g1)}  (+{int((per_season>0).sum())}/{len(per_season)})  "
          + "  ".join(f"{s}:{v:+.3f}" for s, v in per_season.items()))
    worst = min(without, key=without.get)
    print(f"  GATE 2  drop best league       {_paint(g2)}  (drop {worst} -> {without[worst]:+.4f})")
    wf_detail = "  ".join(f"{f}:{v:+.3f}" for f, v in wf.items()) if not walk.empty else "no bets"
    print(f"  GATE 3  walk-forward           {_paint(g3)}  ({wf_detail})")
    print(f"  diagnostic spread              {spread:.3f}")

    from evaluation.results_registry import upsert
    upsert(source="goals_foragst_favourite_draw", space="goals_foragst", target="draw",
           abt_filter="major, gn>8, dropna", segmentation="mrkt_favourite",
           segment_filter=f"most-profitable seg (raw) = {BEST_SEGMENT}",
           g1=_verdict(g1), g2=_verdict(g2), g3=_verdict(g3),
           result=f"pooled {pooled:+.3f} ({len(bets)} bets)", roi_std=f"{spread:.3f}")
    print("\nlogged to model_checks/results.md")

PART 2 - LR on segment 'home', draw market (b365):
  pooled ROI                     +0.0101  (563 bets)
  GATE 1  LOSO (>=3/4)            FAIL  (+2/4)  2223:-0.236  2324:+0.339  2425:-0.023  2526:+0.030
  GATE 2  drop best league       FAIL  (drop england -> -0.0520)
  GATE 3  walk-forward           PASS  (2425:-0.014  2526:+0.030)
  diagnostic spread              0.420

logged to model_checks/results.md
